## Weighting & Aggregation

## 1. loading normalised SLQI dataset

In [38]:
import pandas as pd
import numpy as np

normalised_slqi_df = pd.read_csv("../Dataset/normalised_SLQI_dataset.csv")
normalised_slqi_df.head()

,Country,Country Code,gdp_per_capita,inflation,unemployment_rate,life_expectancy,secondary_enrollment,electricity_access,drinking_water_access,sanitation_access,...,drinking_water_access_normalised,sanitation_access_normalised,government_effectiveness_normalised,political_stability_normalised,rule_of_law_normalised,forest_area_normalised,renewable_energy_normalised,inflation_normalised,unemployment_rate_normalised,pm25_exposure_normalised
0,Afghanistan,AFG,413.757895,-6.601186,13.6870,66.289,59.613602,85.3,80.832437,54.495799,...,0.701991,0.492380,0.053644,0.125863,0.063870,0.019617,0.207684,0.975468,0.607191,0.486559
1,Albania,ALB,11377.775743,2.215874,10.6890,79.776,108.355392,100.0,95.119427,99.299715,...,0.924119,0.992188,0.558159,0.639530,0.493751,0.304846,0.435099,0.937493,0.694057,0.865235
2,Algeria,DZA,5752.990767,4.046115,11.6550,76.475,105.164132,100.0,92.420029,85.906486,...,0.882150,0.842781,0.431593,0.474732,0.356937,0.008791,0.001038,0.929611,0.666068,0.742513
3,American Samoa,ASM,18017.458938,3.117464,5.1895,72.992,92.659564,100.0,100.000000,62.119698,...,1.000000,0.577428,0.660358,0.923995,0.778862,0.902088,0.004154,0.933610,0.853403,0.977315
4,Andorra,AND,49303.649167,3.117464,5.1895,84.188,101.923820,100.0,100.000000,100.000000,...,1.000000,1.000000,0.748322,0.991101,0.855357,0.360439,0.194185,0.933610,0.853403,0.947834


## 2. Define normalised columns by dimension

In [39]:
economic_cols = [
    "gdp_per_capita_normalised",
    "inflation_normalised",
    "unemployment_rate_normalised"
]

health_education_cols = [
    "life_expectancy_normalised",
    "secondary_enrollment_normalised"
]

infrastructure_cols = [
    "electricity_access_normalised",
    "drinking_water_access_normalised",
    "sanitation_access_normalised"
]

governance_cols = [
    "government_effectiveness_normalised",
    "political_stability_normalised",
    "rule_of_law_normalised"
]

environment_cols = [
    "forest_area_normalised",
    "pm25_exposure_normalised",
    "renewable_energy_normalised"
]

In [40]:
normalised_slqi_df["economic_score"] = normalised_slqi_df[economic_cols].mean(axis=1)
normalised_slqi_df["health_education_score"] = normalised_slqi_df[health_education_cols].mean(axis=1)
normalised_slqi_df["infrastructure_score"] = normalised_slqi_df[infrastructure_cols].mean(axis=1)
normalised_slqi_df["governance_score"] = normalised_slqi_df[governance_cols].mean(axis=1)
normalised_slqi_df["environment_score"] = normalised_slqi_df[environment_cols].mean(axis=1)

In [41]:
normalised_slqi_df[[ "Country", "economic_score", "health_education_score", "infrastructure_score", "governance_score", "environment_score"]].head(200)

,Country,economic_score,health_education_score,infrastructure_score,governance_score,environment_score
0,Afghanistan,0.527778,0.364152,0.679660,0.081126,0.237953
1,Albania,0.556775,0.732830,0.972102,0.563813,0.535060
2,Algeria,0.538302,0.670747,0.908310,0.421087,0.250781
3,American Samoa,0.616286,0.575804,0.859143,0.787738,0.627852
4,Andorra,0.652525,0.781345,1.000000,0.864927,0.500819
...,...,...,...,...,...,...
195,United States,0.704848,0.683872,0.998479,0.718548,0.478458
196,Uruguay,0.591458,0.744095,0.994939,0.773170,0.552043
197,Uzbekistan,0.596980,0.581990,0.973121,0.483644,0.254515
198,Vanuatu,0.589735,0.690153,0.603891,0.594163,0.509753


## 4. Apply equal weighting

In [42]:
equal_weights = {
    "economic_score": 0.20,
    "health_education_score": 0.20,
    "infrastructure_score": 0.20,
    "governance_score": 0.20,
    "environment_score": 0.20
}

Equal weighting was used because there was no strong theoretical evidence to justify giving one dimension more importance than another.

In [43]:
normalised_slqi_df["slqi_score"] = (
    normalised_slqi_df["economic_score"] * equal_weights["economic_score"] +
    normalised_slqi_df["health_education_score"] * equal_weights["health_education_score"] +
    normalised_slqi_df["infrastructure_score"] * equal_weights["infrastructure_score"] +
    normalised_slqi_df["governance_score"] * equal_weights["governance_score"] +
    normalised_slqi_df["environment_score"] * equal_weights["environment_score"]
)

In [44]:
normalised_slqi_df["slqi_score"].describe()

count    206.000000
mean       0.607236
std        0.118135
min        0.273919
25%        0.523591
50%        0.627284
75%        0.692023
max        0.843708
Name: slqi_score, dtype: float64

In [45]:
normalised_slqi_df["slqi_rank"] = normalised_slqi_df["slqi_score"].rank( ascending=False,method="min").astype(int)

- this means The higher the score, the higher the ranking.

In [56]:
# the top 20:
normalised_slqi_df[
    [
        "Country",
        "slqi_score",
        "slqi_rank",
        "economic_score",
        "health_education_score",
        "infrastructure_score",
        "governance_score",
        "environment_score"
    ]
].sort_values("slqi_rank").head(20)

,Country,slqi_score,slqi_rank,economic_score,health_education_score,infrastructure_score,governance_score,environment_score
107,Liechtenstein,0.866357,1,0.834928,0.835668,0.999814,0.932930,0.615199
123,Monaco,0.882907,2,0.929004,1.000000,1.000000,0.879639,0.387073
63,Finland,0.878255,3,0.628200,0.882677,0.997947,0.907696,0.767288
177,Sweden,0.869694,4,0.630972,0.876169,0.994087,0.863665,0.773135
139,Norway,0.847282,5,0.707388,0.811397,0.992738,0.909410,0.659276
83,Iceland,0.830770,6,0.706495,0.782229,0.995456,0.892555,0.619545
109,Luxembourg,0.822856,7,0.744847,0.787200,0.991080,0.922393,0.510186
178,Switzerland,0.821551,8,0.727003,0.783192,0.999628,0.900505,0.526194
50,Denmark,0.828170,9,0.678564,0.827358,0.998502,0.908627,0.508411
92,Japan,0.820416,10,0.659566,0.777719,0.995556,0.910984,0.572193


In [47]:
# the last 20:
normalised_slqi_df[
    [
        "Country",
        "slqi_score",
        "slqi_rank",
        "economic_score",
        "health_education_score",
        "infrastructure_score",
        "governance_score",
        "environment_score"
    ]
].sort_values("slqi_rank").tail(20)

,Country,slqi_score,slqi_rank,economic_score,health_education_score,infrastructure_score,governance_score,environment_score
184,Togo,0.440103,187,0.628386,0.323780,0.395479,0.376019,0.476849
104,Lesotho,0.438596,188,0.484559,0.337369,0.542170,0.447777,0.381102
179,Syrian Arab Republic,0.437636,189,0.517732,0.395303,0.912503,0.097779,0.264864
20,Benin,0.431855,190,0.634257,0.228824,0.401752,0.470786,0.423654
111,Madagascar,0.426917,191,0.610248,0.243549,0.240306,0.376995,0.663487
79,Haiti,0.420726,192,0.472709,0.452255,0.426498,0.178431,0.573734
30,Burundi,0.403378,193,0.612412,0.278022,0.306010,0.264251,0.556192
137,Nigeria,0.402002,194,0.574136,0.140289,0.577563,0.239374,0.478650
115,Mali,0.395767,195,0.618438,0.209447,0.574736,0.173904,0.402310
203,"Yemen, Rep.",0.391690,196,0.490991,0.520048,0.675559,0.046426,0.225425


# Sensitivity Analysis

## Scenario 2: Health and education more important

In [ ]:
health_priority_weights = {
    "economic_score": 0.10,
    "health_education_score": 0.30,
    "infrastructure_score": 0.30,
    "governance_score": 0.15,
    "environment_score": 0.15
}

In [53]:
normalised_slqi_df["slqi_social_focus"] = (
    normalised_slqi_df["economic_score"] * health_priority_weights["economic_score"] +
    normalised_slqi_df["health_education_score"] * health_priority_weights["health_education_score"] +
    normalised_slqi_df["infrastructure_score"] * health_priority_weights["infrastructure_score"] +
    normalised_slqi_df["governance_score"] * health_priority_weights["governance_score"] +
    normalised_slqi_df["environment_score"] * health_priority_weights["environment_score"]
)

In [ ]:
normalised_slqi_df["rank_social_focus"] = normalised_slqi_df["slqi_social_focus"].rank(ascending=False,method="min").astype(int)

In [57]:
normalised_slqi_df[
    [
        "Country",
        "slqi_social_focus",
        "rank_social_focus",
        "economic_score",
        "health_education_score",
        "infrastructure_score",
        "governance_score",
        "environment_score"
    ]
].sort_values("rank_social_focus").head(20)

,Country,slqi_social_focus,rank_social_focus,economic_score,health_education_score,infrastructure_score,governance_score,environment_score
123,Monaco,0.882907,1,0.929004,1.000000,1.000000,0.879639,0.387073
63,Finland,0.878255,2,0.628200,0.882677,0.997947,0.907696,0.767288
177,Sweden,0.869694,3,0.630972,0.876169,0.994087,0.863665,0.773135
107,Liechtenstein,0.866357,4,0.834928,0.835668,0.999814,0.932930,0.615199
139,Norway,0.847282,5,0.707388,0.811397,0.992738,0.909410,0.659276
83,Iceland,0.830770,6,0.706495,0.782229,0.995456,0.892555,0.619545
50,Denmark,0.828170,7,0.678564,0.827358,0.998502,0.908627,0.508411
109,Luxembourg,0.822856,8,0.744847,0.787200,0.991080,0.922393,0.510186
134,New Zealand,0.821781,9,0.657186,0.786813,1.000000,0.906661,0.560128
178,Switzerland,0.821551,10,0.727003,0.783192,0.999628,0.900505,0.526194


## Scenario 3: Environment more important

In [58]:
environment_priority_weights = {
    "economic_score": 0.15,
    "health_education_score": 0.15,
    "infrastructure_score": 0.15,
    "governance_score": 0.15,
    "environment_score": 0.40
}

In [59]:
normalised_slqi_df["slqi_environment_focus"] = (
    normalised_slqi_df["economic_score"] * environment_priority_weights["economic_score"] +
    normalised_slqi_df["health_education_score"] * environment_priority_weights["health_education_score"] +
    normalised_slqi_df["infrastructure_score"] * environment_priority_weights["infrastructure_score"] +
    normalised_slqi_df["governance_score"] * environment_priority_weights["governance_score"] +
    normalised_slqi_df["environment_score"] * environment_priority_weights["environment_score"]
)

In [61]:
normalised_slqi_df["rank_environment_focus"] = normalised_slqi_df["slqi_environment_focus"].rank( ascending=False,method="min").astype(int)

In [64]:
normalised_slqi_df[
    [
        "Country",
        "slqi_environment_focus",
        "rank_environment_focus",
        "economic_score",
        "health_education_score",
        "infrastructure_score",
        "governance_score",
        "environment_score"
    ]
].sort_values("rank_environment_focus").head(40)

,Country,slqi_environment_focus,rank_environment_focus,economic_score,health_education_score,infrastructure_score,governance_score,environment_score
63,Finland,0.819393,1,0.628200,0.882677,0.997947,0.907696,0.767288
177,Sweden,0.813988,2,0.630972,0.876169,0.994087,0.863665,0.773135
107,Liechtenstein,0.786580,3,0.834928,0.835668,0.999814,0.932930,0.615199
139,Norway,0.776850,4,0.707388,0.811397,0.992738,0.909410,0.659276
83,Iceland,0.754328,5,0.706495,0.782229,0.995456,0.892555,0.619545
22,Bhutan,0.749714,6,0.618789,0.559169,0.948723,0.758181,0.792463
59,Estonia,0.736410,7,0.607962,0.725874,0.996361,0.816345,0.661071
92,Japan,0.730451,8,0.659566,0.777719,0.995556,0.910984,0.572193
134,New Zealand,0.726650,9,0.657186,0.786813,1.000000,0.906661,0.560128
123,Monaco,0.726126,10,0.929004,1.000000,1.000000,0.879639,0.387073


## Scenario 4: Environment more important

In [65]:
governance_priority_weights = {
    "economic_score": 0.15,
    "health_education_score": 0.15,
    "infrastructure_score": 0.15,
    "governance_score": 0.40,
    "environment_score": 0.15
}

In [66]:
normalised_slqi_df["slqi_governance_focus"] = (
    normalised_slqi_df["economic_score"] * governance_priority_weights["economic_score"] +
    normalised_slqi_df["health_education_score"] * governance_priority_weights["health_education_score"] +
    normalised_slqi_df["infrastructure_score"] * governance_priority_weights["infrastructure_score"] +
    normalised_slqi_df["governance_score"] * governance_priority_weights["governance_score"] +
    normalised_slqi_df["environment_score"] * governance_priority_weights["environment_score"]
)

In [67]:
normalised_slqi_df["rank_governance_focus"] = normalised_slqi_df["slqi_governance_focus"].rank( ascending=False,method="min").astype(int)

In [70]:
normalised_slqi_df[
    [
        "Country",
        "slqi_governance_focus",
        "rank_governance_focus",
        "economic_score",
        "health_education_score",
        "infrastructure_score",
        "governance_score",
        "environment_score"
    ]
].sort_values("rank_governance_focus").head(20)

,Country,slqi_governance_focus,rank_governance_focus,economic_score,health_education_score,infrastructure_score,governance_score,environment_score
107,Liechtenstein,0.866013,1,0.834928,0.835668,0.999814,0.932930,0.615199
63,Finland,0.854495,2,0.628200,0.882677,0.997947,0.907696,0.767288
123,Monaco,0.849267,3,0.929004,1.000000,1.000000,0.879639,0.387073
139,Norway,0.839384,4,0.707388,0.811397,0.992738,0.909410,0.659276
177,Sweden,0.836620,5,0.630972,0.876169,0.994087,0.863665,0.773135
109,Luxembourg,0.823954,6,0.744847,0.787200,0.991080,0.922393,0.510186
83,Iceland,0.822581,7,0.706495,0.782229,0.995456,0.892555,0.619545
178,Switzerland,0.815604,8,0.727003,0.783192,0.999628,0.900505,0.526194
50,Denmark,0.815376,9,0.678564,0.827358,0.998502,0.908627,0.508411
92,Japan,0.815149,10,0.659566,0.777719,0.995556,0.910984,0.572193


In [ ]:
normalised_slqi_df["rank_change_environment_focus"] = (
    normalised_slqi_df["rank_environment_focus"] - normalised_slqi_df["slqi_rank"]
).abs()

normalised_slqi_df["rank_change_social_focus"] = (
    normalised_slqi_df["rank_social_focus"] - normalised_slqi_df["slqi_rank"]
).abs()

normalised_slqi_df["rank_change_governance_focus"] = (
    normalised_slqi_df["rank_governance_focus"] - normalised_slqi_df["slqi_rank"]
).abs()

## rank change comparison

In [84]:
normalised_slqi_df["rank_change_social_focus"] = (normalised_slqi_df["rank_social_focus"] - normalised_slqi_df["slqi_rank"]).abs()
normalised_slqi_df[
    [
        "Country",
        "slqi_score",
        "slqi_rank",
        "slqi_social_focus",
        "rank_social_focus",
        "rank_change_social_focus",
        "economic_score",
        "health_education_score",
        "infrastructure_score",
        "governance_score",
        "environment_score"
    ]
].sort_values("rank_change_social_focus", ascending=False).head(20)

,Country,slqi_score,slqi_rank,slqi_social_focus,rank_social_focus,rank_change_social_focus,economic_score,health_education_score,infrastructure_score,governance_score,environment_score
166,Solomon Islands,0.589310,114,0.589310,148,34,0.632102,0.539742,0.543965,0.552957,0.786959
121,"Micronesia, Fed. Sts.",0.684069,68,0.684069,96,28,0.596953,0.487376,0.912732,0.726767,0.635511
7,Argentina,0.655823,143,0.655823,115,28,0.281460,0.688829,0.958891,0.529074,0.360002
158,Saudi Arabia,0.692789,105,0.692789,80,25,0.654294,0.727806,0.986132,0.619726,0.134797
14,Bahrain,0.690385,109,0.690385,85,24,0.672571,0.727936,0.999585,0.585160,0.113986
3,American Samoa,0.704451,50,0.704451,73,23,0.616286,0.575804,0.859143,0.787738,0.627852
190,Tuvalu,0.666194,86,0.666194,109,23,0.602766,0.479387,0.955083,0.705783,0.464727
99,Kuwait,0.705441,94,0.705441,72,22,0.662974,0.757658,1.000000,0.613565,0.132077
117,Marshall Islands,0.639591,102,0.639591,122,20,0.604366,0.460709,0.856690,0.690585,0.535645
187,Tunisia,0.644143,140,0.644143,120,20,0.496344,0.634165,0.977629,0.430390,0.309415


In [ ]:
print("Average rank change:", normalised_slqi_df["rank_change_social_focus"].mean())
print("Maximum rank change:", normalised_slqi_df["rank_change_social_focus"].max())

Average rank change: 7.611650485436893
Maximum rank change: 34


- The sensitivity analysis shows that changing the weighting method does affect the SLQI rankings, but the overall impact is moderate. Under the social-focused weighting scenario, the average rank change was about 7.61 places, while the maximum rank change was 34 places. This suggests that the index is generally stable, but some countries are more sensitive to weighting changes, especially those with uneven performance across the five dimensions.

## Scenario 2: Environment focus VS slqi_rank

In [ ]:
normalised_slqi_df["rank_change_environment_focus"] = (normalised_slqi_df["rank_environment_focus"] - normalised_slqi_df["slqi_rank"]).abs()
normalised_slqi_df[
    [
        "Country",
        "slqi_score",
        "slqi_rank",
        "slqi_environment_focus",
        "rank_environment_focus",
        "rank_change_environment_focus",
        "economic_score",
        "health_education_score",
        "infrastructure_score",
        "governance_score",
        "environment_score"
    ]
].sort_values("rank_change_environment_focus", ascending=False).head(20)

,Country,slqi_score,slqi_rank,slqi_environment_focus,rank_environment_focus,rank_change_environment_focus,economic_score,health_education_score,infrastructure_score,governance_score,environment_score
66,Gabon,0.615597,120,0.665469,35,85,0.463819,0.505485,0.761725,0.393297,0.867049
151,Qatar,0.716596,78,0.501107,160,82,0.735766,0.747647,0.999707,0.752789,0.039302
166,Solomon Islands,0.589310,114,0.655098,47,67,0.632102,0.539742,0.543965,0.552957,0.786959
99,Kuwait,0.705441,94,0.507960,154,60,0.662974,0.757658,1.000000,0.613565,0.132077
193,United Arab Emirates,0.741482,60,0.567088,118,58,0.684995,0.764449,0.996605,0.742384,0.222057
14,Bahrain,0.690385,109,0.493382,164,55,0.672571,0.727936,0.999585,0.585160,0.113986
158,Saudi Arabia,0.692789,105,0.502112,159,54,0.654294,0.727806,0.986132,0.619726,0.134797
43,"Congo, Rep.",0.580746,149,0.592662,96,53,0.456301,0.466096,0.787291,0.352228,0.708436
105,Liberia,0.412282,175,0.546680,126,49,0.611062,0.231467,0.367354,0.368861,0.774670
140,Oman,0.690884,97,0.521806,145,48,0.641722,0.697572,0.959488,0.674379,0.189581


In [83]:
print("Average rank change - Environment focus:",
      normalised_slqi_df["rank_change_environment_focus"].mean())

print("Maximum rank change - Environment focus:",
      normalised_slqi_df["rank_change_environment_focus"].max())

Average rank change - Environment focus: 16.349514563106798
Maximum rank change - Environment focus: 85


## Scenario 3: Governance focus VS slqi_rank

In [85]:
normalised_slqi_df["rank_change_governance_focus"] = (normalised_slqi_df["rank_governance_focus"] - normalised_slqi_df["slqi_rank"]).abs()
normalised_slqi_df[
    [
        "Country",
        "slqi_score",
        "slqi_rank",
        "slqi_governance_focus",
        "rank_governance_focus",
        "rank_change_governance_focus",
        "economic_score",
        "health_education_score",
        "infrastructure_score",
        "governance_score",
        "environment_score"
    ]
].sort_values("rank_change_governance_focus", ascending=False).head(20)


,Country,slqi_score,slqi_rank,slqi_governance_focus,rank_governance_focus,rank_change_governance_focus,economic_score,health_education_score,infrastructure_score,governance_score,environment_score
25,Botswana,0.579185,146,0.588908,112,34,0.424920,0.446782,0.769100,0.683438,0.462752
96,Kiribati,0.571711,138,0.598739,106,32,0.599018,0.467826,0.673203,0.672926,0.457076
31,Cabo Verde,0.658182,115,0.623507,85,30,0.539755,0.638675,0.892320,0.671943,0.294114
40,Colombia,0.701226,85,0.586917,114,29,0.556796,0.681746,0.967662,0.421051,0.583774
130,Nauru,0.595298,136,0.596379,109,27,0.611180,0.403995,0.882179,0.653796,0.335052
128,Myanmar,0.538521,160,0.416744,186,26,0.619229,0.483264,0.748077,0.127854,0.586780
72,Greenland,0.703844,61,0.728212,36,25,0.663175,0.533142,0.973938,0.869085,0.366932
199,"Venezuela, RB",0.632242,139,0.462849,163,24,0.599032,0.587329,0.959475,0.130702,0.591281
145,Paraguay,0.686857,81,0.604716,103,22,0.597023,0.547753,0.982124,0.469957,0.651320
146,Peru,0.685572,92,0.588315,113,21,0.606709,0.703820,0.885068,0.442965,0.545266


In [86]:
print("Average rank change - Governance focus:", normalised_slqi_df["rank_change_governance_focus"].mean())
print("Maximum rank change - Governance focus:", normalised_slqi_df["rank_change_governance_focus"].max())

Average rank change - Governance focus: 7.495145631067961
Maximum rank change - Governance focus: 34


In [87]:
sensitivity_summary = pd.DataFrame({
    "Scenario": [
        "Social focus",
        "Environment focus",
        "Governance focus"
    ],
    "Average rank change": [
        normalised_slqi_df["rank_change_social_focus"].mean(),
        normalised_slqi_df["rank_change_environment_focus"].mean(),
        normalised_slqi_df["rank_change_governance_focus"].mean()
    ],
    "Maximum rank change": [
        normalised_slqi_df["rank_change_social_focus"].max(),
        normalised_slqi_df["rank_change_environment_focus"].max(),
        normalised_slqi_df["rank_change_governance_focus"].max()
    ]
})

sensitivity_summary

,Scenario,Average rank change,Maximum rank change
0,Social focus,7.611650,34
1,Environment focus,16.349515,85
2,Governance focus,7.495146,34


- The sensitivity analysis shows that the SLQI results are relatively stable under the social focus and governance focus scenarios. The average rank changes were 7.61 and 7.50 places respectively, with a maximum rank change of 34 places in both cases.
- However, the environment focus scenario had a stronger impact on the rankings. The average rank change increased to 16.35 places, and the maximum rank change reached 85 places. This suggests that the SLQI is more sensitive to changes in the environmental weighting. 

In [89]:
bottom_10 = normalised_slqi_df[
    [
        "Country",
        "slqi_score",
        "slqi_rank",
        "economic_score",
        "health_education_score",
        "infrastructure_score",
        "governance_score",
        "environment_score"
    ]
].sort_values("slqi_rank", ascending=False).head(10)

bottom_10

,Country,slqi_score,slqi_rank,economic_score,health_education_score,infrastructure_score,governance_score,environment_score
169,South Sudan,0.255144,206,0.408578,0.431342,0.036219,0.115372,0.378084
37,Chad,0.213604,205,0.628181,0.074364,0.119263,0.215028,0.402961
136,Niger,0.245004,204,0.634210,0.159896,0.165997,0.280336,0.278432
167,"Somalia, Fed. Rep.",0.314089,203,0.463254,0.067981,0.490305,0.054182,0.614339
29,Burkina Faso,0.287058,202,0.611901,0.190300,0.211354,0.266345,0.436132
36,Central African Republic,0.298454,201,0.588098,0.335376,0.060930,0.153929,0.651088
42,"Congo, Dem. Rep.",0.304705,200,0.603591,0.284710,0.080953,0.130113,0.767532
0,Afghanistan,0.413783,199,0.527778,0.364152,0.679660,0.081126,0.237953
118,Mauritania,0.395366,198,0.549817,0.326205,0.553494,0.381559,0.128273
175,Sudan,0.430373,197,0.379146,0.474256,0.570736,0.118316,0.408092


In [90]:
top_10 = normalised_slqi_df[
    [
        "Country",
        "slqi_score",
        "slqi_rank",
        "economic_score",
        "health_education_score",
        "infrastructure_score",
        "governance_score",
        "environment_score"
    ]
].sort_values("slqi_rank").head(10)

top_10

,Country,slqi_score,slqi_rank,economic_score,health_education_score,infrastructure_score,governance_score,environment_score
107,Liechtenstein,0.866357,1,0.834928,0.835668,0.999814,0.932930,0.615199
123,Monaco,0.882907,2,0.929004,1.000000,1.000000,0.879639,0.387073
63,Finland,0.878255,3,0.628200,0.882677,0.997947,0.907696,0.767288
177,Sweden,0.869694,4,0.630972,0.876169,0.994087,0.863665,0.773135
139,Norway,0.847282,5,0.707388,0.811397,0.992738,0.909410,0.659276
83,Iceland,0.830770,6,0.706495,0.782229,0.995456,0.892555,0.619545
109,Luxembourg,0.822856,7,0.744847,0.787200,0.991080,0.922393,0.510186
178,Switzerland,0.821551,8,0.727003,0.783192,0.999628,0.900505,0.526194
50,Denmark,0.828170,9,0.678564,0.827358,0.998502,0.908627,0.508411
92,Japan,0.820416,10,0.659566,0.777719,0.995556,0.910984,0.572193


In [91]:
dimension_correlation = normalised_slqi_df[
    [
        "slqi_score",
        "economic_score",
        "health_education_score",
        "infrastructure_score",
        "governance_score",
        "environment_score"
    ]
].corr()

dimension_correlation["slqi_score"].sort_values(ascending=False)

slqi_score                1.000000
health_education_score    0.913791
infrastructure_score      0.894537
governance_score          0.829059
economic_score            0.454606
environment_score         0.015618
Name: slqi_score, dtype: float64

The correlation analysis between the final SLQI score and the five dimension scores shows that health and education, infrastructure, and governance have the strongest relationships with the final index. Their correlations with the SLQI score are above 0.80, suggesting that these dimensions are major drivers of the final ranking.

In contrast, the environment score shows a very weak correlation with the final SLQI score. This suggests that environmental performance does not strongly align with the overall SLQI ranking under the equal weighting method. Some highly ranked countries perform very well in health, infrastructure, and governance, but do not necessarily have the highest environmental scores.